# 07 — Flight Recommendation System
**FlightIQ — AI Travel Price Intelligence**

MIC AIML Department Recruitment Challenge  
Track: Data Science & Visualization — AI Travel Analyst

---
### Overview
This notebook demonstrates the **Two-Stage Transparent Flight Recommendation Engine** built using actual flight records from `data/processed/cleaned_flight_data.csv`.

1. **Stage 1 — Multi-criteria Filtering & Graceful Constraint Relaxation**:
   - Hard constraints: Source, Destination, Travel Class.
   - Soft constraints: Budget, Max Stops, Duration, Preferred Airline, Season.
   - In case of over-constrained queries, optional constraints are relaxed intelligently.
2. **Stage 2 — Normalized Multi-factor Transparent Scoring & Ranking**:
   - Price & Budget Efficiency (50%)
   - Flight Stops (20%)
   - Travel Duration (15%)
   - Airline Preference (10%)
   - Timing Window Fit (5%)
3. **Factual 'Why this flight?' Explanations**:
   - Generates truthful, verified reasons for every recommended flight.
   - Every recommendation maps strictly to a real record in the dataset.

In [1]:
import sys, os
import pandas as pd
import numpy as np

# Add src to path
sys.path.insert(0, os.path.abspath('../src'))
from recommender import FlightRecommender, get_flight_recommendations, DEFAULT_WEIGHTS

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

rec = FlightRecommender()
print(f"Recommender initialized with {len(rec.df):,} real flight records.")
print(f"Available Origins ({len(rec.available_sources)}): {rec.available_sources[:6]}...")
print(f"Available Destinations ({len(rec.available_destinations)}): {rec.available_destinations[:6]}...")
print(f"Travel Classes: {rec.available_classes}")
print(f"Airlines ({len(rec.available_airlines)}): {rec.available_airlines[:5]}...")

Recommender initialized with 93,083 real flight records.
Available Origins (18): ['Ahmedabad', 'Bangalore', 'Bangkok', 'Chennai', 'Delhi', 'Doha']...
Available Destinations (18): ['Ahmedabad', 'Bangalore', 'Bangkok', 'Chennai', 'Delhi', 'Doha']...
Travel Classes: ['Business', 'Economy', 'First', 'Premium Economy']
Airlines (13): ['Air India', 'Airasia India', 'British Airways', 'Emirates', 'Etihad Airways']...


## 1. Scenario A: Budget-Conscious Traveler (Domestic Route)
**User Goal**: Find the best economy flight from **Mumbai** to **Goa** under **₹5,000** with **0 stops** and preferred airline **Indigo**.

In [2]:
res_a = rec.recommend(
    source="Mumbai",
    destination="Goa",
    travel_class="Economy",
    max_budget=5000,
    max_stops=0,
    preferred_airline="Indigo",
    top_k=5
)

print(f"Status: {res_a['status']} | Found {res_a['count']} top recommendations out of {res_a['total_matching_candidates']} candidates.")
if res_a['relaxation_notes']:
    print(f"Relaxation Notes: {res_a['relaxation_notes']}")

for flight in res_a['recommendations']:
    print("-" * 80)
    print(f"Rank #{flight['rank']}: {flight['airline']} | {flight['source']} → {flight['destination']} | {flight['travel_class']}")
    print(f"Price: {flight['price_formatted']} | Stops: {flight['stops_formatted']} | Duration: {flight['duration_formatted']}")
    print(f"Departure: {flight['departure_time']} | Arrival: {flight['arrival_time']} | Match Score: {flight['match_score']}/100")
    print("Why this flight:")
    for reason in flight['why_this_flight']:
        print(f"  ✓ {reason}")

Status: SUCCESS | Found 5 top recommendations out of 137 candidates.
--------------------------------------------------------------------------------
Rank #1: Gofirst | Mumbai → Goa | Economy
Price: ₹342.98 | Stops: Non-Stop | Duration: 1h 03m
Departure: 5:30 AM | Arrival: 6:33 AM | Match Score: 88.1/100
Why this flight:
  ✓ Within your budget (saves ₹4,657 from max budget)
  ✓ Lowest price flight among all available options on this route
  ✓ Direct non-stop flight for maximum convenience
  ✓ Shorter journey time (1h 03m)
--------------------------------------------------------------------------------
Rank #2: Airasia India | Mumbai → Goa | Economy
Price: ₹704.40 | Stops: Non-Stop | Duration: 1h 03m
Departure: 10:00 | Arrival: 11:03 | Match Score: 84.3/100
Why this flight:
  ✓ Within your budget (saves ₹4,296 from max budget)
  ✓ Highly competitive fare (in the cheapest 25% of available flights)
  ✓ Direct non-stop flight for maximum convenience
  ✓ Shorter journey time (1h 03m)
------

## 2. Scenario B: Business / Premium Traveler (International Route)
**User Goal**: Find a **Business Class** flight from **Delhi** to **London** with preferred airline **British Airways** and departure in the morning.

In [3]:
res_b = rec.recommend(
    source="Delhi",
    destination="London",
    travel_class="Business",
    preferred_airline="British Airways",
    preferred_departure_time="09:00 AM",
    top_k=5
)

print(f"Status: {res_b['status']} | Found {res_b['count']} recommendations.")
for flight in res_b['recommendations']:
    print("-" * 80)
    print(f"Rank #{flight['rank']}: {flight['airline']} | {flight['source']} → {flight['destination']} | {flight['travel_class']}")
    print(f"Price: {flight['price_formatted']} | Stops: {flight['stops_formatted']} | Duration: {flight['duration_formatted']}")
    print(f"Departure: {flight['departure_time']} | Arrival: {flight['arrival_time']} | Match Score: {flight['match_score']}/100")
    print("Why this flight:")
    for reason in flight['why_this_flight']:
        print(f"  ✓ {reason}")

Status: SUCCESS | Found 5 recommendations.
--------------------------------------------------------------------------------
Rank #1: Qatar Airways | Delhi → London | Business
Price: ₹164,425.78 | Stops: Non-Stop | Duration: 9h 10m
Departure: 11:30 | Arrival: 20:39 | Match Score: 53.6/100
Why this flight:
  ✓ Highly competitive fare (in the cheapest 25% of available flights)
  ✓ Direct non-stop flight for maximum convenience
  ✓ Fastest journey on this route (9h 10m)
--------------------------------------------------------------------------------
Rank #2: Vistara | Delhi → London | Business
Price: ₹78,843.98 | Stops: 2 Stop(s) | Duration: 12h 43m
Departure: 20:05 | Arrival: 08:48 | Match Score: 51.2/100
Why this flight:
  ✓ Lowest price flight among all available options on this route
--------------------------------------------------------------------------------
Rank #3: Etihad Airways | Delhi → London | Business
Price: ₹179,436.76 | Stops: Non-Stop | Duration: 9h 43m
Departure: 11:30

## 3. Scenario C: Graceful Constraint Relaxation (Over-constrained Query)
**User Goal**: Long-haul route **Sydney** to **New York** with an impossibly low budget (**₹10,000**) and **0 stops** (no direct non-stop flight exists).

**Expected Behavior**: The recommender must NOT crash or return an empty error. It must relax impossible constraints gracefully and report them.

In [4]:
res_c = rec.recommend(
    source="Sydney",
    destination="New York",
    travel_class="Economy",
    max_budget=10000,
    max_stops=0,
    top_k=5
)

print(f"Status: {res_c['status']}")
print("Constraint Relaxation Notes:")
for note in res_c['relaxation_notes']:
    print(f"  ℹ {note}")

print(f"\nFallback Top Recommendations ({len(res_c['recommendations'])}): ")
for flight in res_c['recommendations']:
    print(f"  Rank #{flight['rank']}: {flight['airline']} | {flight['price_formatted']} | {flight['stops_formatted']} | {flight['duration_formatted']}")
    print(f"    Why: {'; '.join(flight['why_this_flight'])}")

Status: SUCCESS
Constraint Relaxation Notes:

Fallback Top Recommendations (1): 
  Rank #1: Singapore Airlines | ₹289.37 | Non-Stop | 22h 36m
    Why: Within your budget (saves ₹9,711 from max budget); Lowest price flight among all available options on this route; Direct non-stop flight for maximum convenience; Fastest journey on this route (22h 36m)


## 4. Recommender Verification & Integrity Check
Confirm that every recommended flight exists in the source dataset and has accurate details.

In [5]:
test_recs = res_a['recommendations'] + res_b['recommendations'] + res_c['recommendations']
all_valid = True
for r in test_recs:
    fid = r['flight_id']
    match = rec.df[rec.df['Flight_ID'] == fid]
    if match.empty:
        all_valid = False
        print(f"Flight {fid} not found in raw dataset!")
    else:
        actual_price = float(match.iloc[0]['Price'])
        assert abs(actual_price - r['price']) < 0.01, f"Price mismatch for {fid}"

if all_valid:
    print(f"Integrity check PASSED: All {len(test_recs)} tested recommendations are verified authentic records from the dataset.")

Integrity check PASSED: All 11 tested recommendations are verified authentic records from the dataset.
